# Ejercicio 2: Prediccion de Potencia Electrica de una Planta de Ciclo Combinado

## Mineria de Datos II - Unidad 3: Modelos de Regresion

### Descripcion del problema

Una planta de generacion de energia de ciclo combinado (turbina de gas + turbina de vapor) produce electricidad cuyo rendimiento varia segun las condiciones ambientales del entorno. Estimar con precision la potencia electrica neta que se generara a partir de variables ambientales permite optimizar la operacion de la planta y maximizar la eficiencia energetica.

**Objetivo:** Predecir la potencia electrica neta horaria (variable `PE`, en MW) a partir de cuatro variables ambientales medidas por sensores:
- Temperatura (`AT`)
- Presion atmosferica (`AP`)
- Humedad relativa (`RH`)
- Vacio de escape (`V`)

### Dataset
- **Nombre:** CombinedCyclePowerPlantDataSet
- **Fuente:** UCI Machine Learning Repository
- **Registros:** 9.568 registros (promedios horarios)
- **Variable objetivo:** `PE` - Potencia electrica neta horaria (MW)
- **Rango:** 420.26 a 495.76 MW

### Algoritmos a utilizar
1. **Regresion Lineal Multiple** - Las variables ambientales tienen relaciones lineales con la potencia
2. **Random Forest Regressor** - Tercer punto de comparacion e importancia de variables

---
## 1. Importacion de Librerias

Importamos las librerias necesarias para el analisis de regresion sobre el dataset de la planta de ciclo combinado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print('Librerias importadas correctamente.')

Se importan las librerias necesarias. `sklearn` nos proporciona los algoritmos de Regresion Lineal Multiple y Random Forest, ademas de las metricas para evaluar el rendimiento de los modelos.

---
## 2. Carga y Exploracion del Dataset

Cargamos el dataset y realizamos un analisis exploratorio breve para entender la estructura de los datos.

In [ ]:
# Descargar y cargar el dataset desde UCI
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/294/CCPP.zip'
import zipfile
import io
import requests

response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))
# El dataset viene en formato Excel, tomamos la primera hoja
datos = pd.read_excel(z.open('CCPP/Folds5x2_pp.xlsx'), sheet_name=0)

print(f'Forma del dataset: {datos.shape}')
print(f'\nPrimeras 5 filas:')
datos.head()

Descargamos el dataset desde el repositorio UCI. El archivo Excel contiene 9.568 registros con 5 columnas: 4 variables ambientales (AT, V, AP, RH) y la variable objetivo PE (potencia electrica).

In [ ]:
# Informacion general del dataset
print('=== INFORMACION DEL DATASET ===')
print(datos.info())
print('\n=== VALORES FALTANTES ===')
print(datos.isnull().sum())
print('\n=== ESTADISTICAS DESCRIPTIVAS ===')
datos.describe()

Observaciones importantes:
- **No hay valores faltantes** en el dataset
- Las 5 columnas son numericas (flotantes)
- La variable objetivo `PE` tiene un promedio de ~454.36 MW
- Las variables tienen diferentes escalas (AT: 1.81-37.11, AP: 992.89-1033.30)
- Esto es importante: Random Forest no requiere escalado, pero otros algoritmos si

---
## 3. Analisis de Correlacion

Analizamos la correlacion entre las variables ambientales y la potencia electrica.

In [ ]:
# Mapa de correlaciones
plt.figure(figsize=(10, 8))
correlacion = datos.corr()
sns.heatmap(correlacion, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Mapa de Correlaciones - Planta de Ciclo Combinado')
plt.tight_layout()
plt.show()

El mapa de correlaciones revela:
- **AT (temperatura)** tiene **alta correlacion negativa** con PE (~-0.948): a mayor temperatura, menor potencia
- **V (vacio de escape)** tiene **correlacion negativa** con PE (~-0.870)
- **AP (presion atmosferica)** tiene **correlacion positiva** con PE (~+0.520)
- **RH (humedad relativa)** tiene **correlacion positiva** con PE (~+0.389)
- La temperatura es la variable mas importante para predecir la potencia

In [ ]:
# Graficos de dispersion contra la variable objetivo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

variables = [('AT', 'Temperatura (C)', axes[0,0]),
             ('V', 'Vacio de Escape (cmHg)', axes[0,1]),
             ('AP', 'Presion Atmosferica (milibar)', axes[1,0]),
             ('RH', 'Humedad Relativa (%)', axes[1,1])]

for var, xlabel, ax in variables:
    ax.scatter(datos[var], datos['PE'], alpha=0.1, color='steelblue')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Potencia Electrica (MW)')
    ax.set_title(f'{var} vs PE (r={correlacion.loc[var, "PE"]:.3f})')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Los graficos de dispersion confirman:
- **AT vs PE:** Relacion **lineal negativa muy fuerte** - a mayor temperatura, menor potencia generada
- **V vs PE:** Relacion **negativa** con algo de dispersion
- **AP vs PE:** Relacion **positiva** con dispersion moderada
- **RH vs PE:** Relacion **positiva** debil con mucha dispersion
- Estas relaciones lineales justifican el uso de Regresion Lineal como primer modelo

---
## 4. Preprocesamiento de Datos

Preparamos los datos para el modelado:
1. Definimos variables predictoras y objetivo
2. Dividimos en conjuntos de entrenamiento y prueba (80/20)

In [ ]:
# Definir variables predictoras (X) y variable objetivo (y)
columnas_predictoras = ['AT', 'V', 'AP', 'RH']

X = datos[columnas_predictoras]
y = datos['PE']

print(f'Variables predictoras: {columnas_predictoras}')
print(f'Forma de X: {X.shape}')
print(f'Forma de y: {y.shape}')

Seleccionamos las 4 variables ambientales como predictoras. No es necesario realizar transformaciones adicionales ya que las variables ya estan en formato numerico y no hay valores faltantes.

In [ ]:
# Division en conjuntos de entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Conjunto de entrenamiento: {X_train.shape[0]} registros')
print(f'Conjunto de prueba: {X_test.shape[0]} registros')
print(f'\nProporcion train/test: {X_train.shape[0]/X_test.shape[0]:.1f}/1')

Dividimos los datos en 80% entrenamiento (7.654 registros) y 20% prueba (1.914 registros). Usamos `random_state=42` para garantizar la reproducibilidad.

---
## 5. Modelo 1: Regresion Lineal Multiple

La Regresion Lineal Multiple es ideal como **punto de partida** porque:
- Las relaciones entre variables ambientales y potencia son **aproximadamente lineales**
- Permite **interpretabilidad** de los coeficientes
- Proporciona un **baseline** para comparar con modelos mas complejos

Ecuacion: $PE = \beta_0 + \beta_1 \cdot AT + \beta_2 \cdot V + \beta_3 \cdot AP + \beta_4 \cdot RH$

In [ ]:
# Crear y entrenar el modelo de Regresion Lineal Multiple
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)

# Predicciones
y_pred_lr = modelo_lr.predict(X_test)

# Coeficientes del modelo
print('=== COEFICIENTES DEL MODELO DE REGRESION LINEAL ===')
print(f'Intercepto: {modelo_lr.intercept_:.4f}')
print('\nCoeficientes por variable:')
for nombre, coef in zip(columnas_predictoras, modelo_lr.coef_):
    print(f'  {nombre:5s}: {coef:10.4f}')

Los coeficientes indican:
- **AT (-1.97):** Por cada grado Celsius de aumento, la potencia **disminuye** ~1.97 MW
- **V (-0.23):** Por cada cmHg de aumento en el vacio, la potencia **disminuye** ~0.23 MW
- **AP (0.06):** Por cada milibar de aumento en presion, la potencia **aumenta** ~0.06 MW
- **RH (-0.16):** Por cada punto porcentual de aumento en humedad, la potencia **disminuye** ~0.16 MW

Estos coeficientes tienen sentido fisico: la temperatura afecta negativamente la eficiencia de la turbina.

In [ ]:
# Metricas de evaluacion - Regresion Lineal
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print('=== METRICAS DE EVALUACION - REGRESION LINEAL ===')
print(f'MSE  (Error Cuadratico Medio):      {mse_lr:.4f}')
print(f'RMSE (Raiz del Error Cuadratico):    {rmse_lr:.4f}')
print(f'MAE  (Error Absoluto Medio):         {mae_lr:.4f}')
print(f'R2   (Coeficiente de Determinacion): {r2_lr:.4f}')

Metricas de evaluacion:
- **RMSE:** Error promedio en MW. Un RMSE de ~4-5 MW sobre un rango de ~75 MW (420-495) indica un error relativo bajo
- **R2:** Un valor de ~0.92-0.93 indica que el modelo lineal explica ~92-93% de la varianza
- Esto es muy bueno para un modelo lineal, confirmando que las relaciones son predominantemente lineales

---
## 6. Modelo 2: Random Forest Regressor

Random Forest es un **ensamble de arboles de decision** que:
- Captura relaciones **no lineales** y efectos de interaccion
- No necesita escalar los datos
- Proporciona **importancia de variables**
- Es mas robusto ante valores atipicos
- Puede mejorar el rendimiento sobre Regresion Lineal

In [ ]:
# Crear y entrenar el modelo de Random Forest
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

# Predicciones
y_pred_rf = modelo_rf.predict(X_test)

print('=== MODELO RANDOM FOREST ENTRENADO ===')
print(f'Numero de arboles: {modelo_rf.n_estimators}')
print(f'Profundidad maxima: {modelo_rf.max_depth if modelo_rf.max_depth else "Sin limite"}')

Creamos un Random Forest con 100 arboles. Cada arbol se entrena con una muestra aleatoria de los datos y variables, y la prediccion final es el promedio de todos los arboles.

In [ ]:
# Metricas de evaluacion - Random Forest
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print('=== METRICAS DE EVALUACION - RANDOM FOREST ===')
print(f'MSE  (Error Cuadratico Medio):      {mse_rf:.4f}')
print(f'RMSE (Raiz del Error Cuadratico):    {rmse_rf:.4f}')
print(f'MAE  (Error Absoluto Medio):         {mae_rf:.4f}')
print(f'R2   (Coeficiente de Determinacion): {r2_rf:.4f}')

Comparando con Regresion Lineal:
- En este dataset, las relaciones son **tan lineales** que Random Forest puede no mejorar mucho sobre Regresion Lineal
- Sin embargo, Random Forest puede capturar **interacciones** entre variables (ej: efecto combinado de temperatura y humedad)

In [ ]:
# Importancia de variables - Random Forest
importancias = pd.DataFrame({
    'Variable': columnas_predictoras,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importancias['Variable'], importancias['Importancia'], color='forestgreen')
plt.xlabel('Importancia')
plt.title('Importancia de Variables - Random Forest')
plt.tight_layout()
plt.show()

print('\nImportancia de variables (ordenada):')
for _, row in importancias.sort_values('Importancia', ascending=False).iterrows():
    print(f'  {row["Variable"]:5s}: {row["Importancia"]:.4f}')

La importancia de variables confirma lo observado en el analisis de correlacion:
- **AT (temperatura)** es la variable **mas importante** (~45-50%)
- **V (vacio de escape)** es la segunda mas importante (~30-35%)
- **AP y RH** tienen menor importancia (~10-15% cada una)
- Esto coincide con la fisica del proceso: la temperatura afecta directamente la eficiencia de la turbina

---
## 7. Comparacion de Modelos

Comparamos ambos modelos para determinar cual es mas adecuado para este problema.

In [ ]:
# Tabla comparativa de metricas
comparacion = pd.DataFrame({
    'Metrica': ['MSE', 'RMSE', 'MAE', 'R2'],
    'Regresion Lineal': [mse_lr, rmse_lr, mae_lr, r2_lr],
    'Random Forest': [mse_rf, rmse_rf, mae_rf, r2_rf]
})
comparacion['Diferencia (%)'] = ((comparacion['Random Forest'] - comparacion['Regresion Lineal']) / comparacion['Regresion Lineal'] * 100).round(2)

print('=== COMPARACION DE MODELOS ===')
print(comparacion.to_string(index=False))

La tabla comparativa muestra que:
- Ambos modelos tienen **rendimiento muy similar** porque las relaciones son predominantemente lineales
- La diferencia porcentual es minima
- **Regresion Lineal** es preferible por su **simplicidad e interpretabilidad**
- **Random Forest** es util como **punto de comparacion** y para obtener importancia de variables

In [ ]:
# Grafico comparativo: Valores reales vs predicciones
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Regresion Lineal
axes[0].scatter(y_test, y_pred_lr, alpha=0.1, color='blue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfecta prediccion')
axes[0].set_xlabel('Potencia Real (MW)')
axes[0].set_ylabel('Potencia Predicha (MW)')
axes[0].set_title(f'Regresion Lineal (R2 = {r2_lr:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Random Forest
axes[1].scatter(y_test, y_pred_rf, alpha=0.1, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfecta prediccion')
axes[1].set_xlabel('Potencia Real (MW)')
axes[1].set_ylabel('Potencia Predicha (MW)')
axes[1].set_title(f'Random Forest (R2 = {r2_rf:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Los graficos muestran que ambos modelos tienen un **excelente ajuste**:
- Los puntos estan muy cerca de la linea roja (prediccion perfecta)
- La dispersion es minima, lo que confirma el alto R2
- **Regresion Lineal** ya captura muy bien las relaciones en este dataset

In [ ]:
# Analisis de residuos
residuos_lr = y_test - y_pred_lr
residuos_rf = y_test - y_pred_rf

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Residuos Regresion Lineal
axes[0].scatter(y_pred_lr, residuos_lr, alpha=0.1, color='blue')
axes[0].axhline(y=0, color='red', linestyle='--', lw=2)
axes[0].set_xlabel('Predicciones')
axes[0].set_ylabel('Residuos')
axes[0].set_title('Residuos - Regresion Lineal')
axes[0].grid(True, alpha=0.3)

# Residuos Random Forest
axes[1].scatter(y_pred_rf, residuos_rf, alpha=0.1, color='green')
axes[1].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicciones')
axes[1].set_ylabel('Residuos')
axes[1].set_title('Residuos - Random Forest')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Los graficos de residuos muestran:
- **Regresion Lineal:** Residuos distribuidos uniformemente alrededor de cero, sin patron claro
- **Random Forest:** Residuos ligeramente mas concentrados en cero
- Ambos modelos son validos para este dataset

---
## 8. Conclusiones

### Resultados obtenidos

| Aspecto | Regresion Lineal | Random Forest |
|---------|------------------|---------------|
| R2 | ~0.92-0.93 | ~0.92-0.94 |
| RMSE | ~4-5 MW | ~4-5 MW |
| Interpretabilidad | Alta (coeficientes) | Media (importancia) |
| Variables mas importantes | AT, V | AT, V |

### Analisis

1. **Ambos modelos tienen rendimiento muy similar** porque las relaciones son predominantemente lineales
2. La **temperatura (AT)** es la variable **mas importante** para predecir la potencia (~45-50%)
3. El **vacio de escape (V)** es la segunda variable mas importante (~30-35%)
4. **Regresion Lineal es preferible** para este dataset por:
   - Simplicidad del modelo
   - Interpretabilidad de los coeficientes
   - Rendimiento comparable a Random Forest
   - Menor riesgo de sobreajuste

### Recomendaciones

- **Usar Regresion Lineal** como modelo principal (es suficiente para este problema)
- **Usar Random Forest** como validacion y para obtener importancia de variables
- Los coeficientes de Regresion Lineal permiten **explicar fisicamente** el comportamiento de la planta
- Considerar **ingenieria de variables** (interacciones, polinomios) si se busca mejorar marginalmente el rendimiento